In [4]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tqdm import tqdm
import numpy as np
import pynapple as nap
import pandas as pd

from Utils.load_files import get_interval_pairs
from Utils.json_tools import read_formatted_json
from Utils.tuning_curve_utils import get_exposure_timestamps, tuning_curve
from Utils.recording import gen_recording_interval_table

file_names = read_formatted_json("./file_names.json")
session_info_filename: str = file_names["session_info_filename"]
head_direction_filename: str = file_names["head_direction_filename"]
interval_table_filename: str = file_names["interval_table_filename"]
kilosort_info_filename: str = file_names["kilosort_info_filename"]


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [5]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m19"

date: str | int = "260831"
multi_recording: bool = True
num_of_rec_list: list[int] = [2, ]
headplate_name: str = 'hp4'
probe_name: str = "A"

phase_key = "baseline"
num_of_bins_in_hd: int = 180
num_shuffle: int = 1000
shuffle_seed: int = 0

camera_input_channel: int = 1

for num_of_rec in num_of_rec_list:

    with tqdm(total=5, desc="Reading files", unit="%",
              bar_format="{l_bar}{bar}| {n}/{total} [{percentage:3.0f}%]") as pbar:
        subfolder_filler = f"{date}_{num_of_rec}"
        base_dir = f"{base_dir}/{date}/{subfolder_filler}" if multi_recording else f"{base_dir}/{date}"

        gen_recording_interval_table(base_dir=base_dir, multi_recording=multi_recording,
                                     camera_input_channel=camera_input_channel, )

        data_dir: str = f"{base_dir}/data"
        kilosort_dir = next((Path(base_dir) / "kilosort" / f"Probe{probe_name}").glob("kilosort_*"))

        session_info: dict = read_formatted_json(f"{data_dir}/{session_info_filename}.json")["session_info"]
        pbar.update(1)

        interval_table = pd.read_csv(f"{data_dir}/{interval_table_filename}.csv")
        interval_pairs_all = np.asarray(
            get_interval_pairs(interval_table,
                               phase_key=phase_key), dtype=float)
        pbar.update(1)

        hd_content = read_formatted_json(f"{data_dir}/processed/{head_direction_filename}.json")
        headplate_data = hd_content[headplate_name]
        hd_raw = headplate_data.get("head_direction_deg", headplate_data.get("hd"))
        if hd_raw is None:
            raise KeyError('Expected "head_direction_deg" or "hd" in head-direction data.')
        hd = np.asarray(hd_raw, dtype=float) % 360
        hd_frames = np.asarray(headplate_data["frames"], dtype=int)
        pbar.update(1)

        exposure_timestamps, adc_time_origin_s, ttl_qc = get_exposure_timestamps(
            session_info=session_info,
            data_dir=data_dir,
            camera_input_channel=camera_input_channel,
            camera_ttl_active_high=False,  # Opto-coupled ExposureActive is active-low.
        )
        pbar.update(1)

        assert len(hd) == len(hd_frames)

        # Basler/DLC frame IDs are 0-based and may be sparse when pose estimates
        # are invalid. Map each valid pose sample to its saved camera timestamp.
        frame_indices = hd_frames
        if not (
            frame_indices.ndim == 1
            and np.all(frame_indices >= 0)
            and np.all(np.diff(frame_indices) > 0)
            and np.all(frame_indices < len(exposure_timestamps))
        ):
            raise ValueError(
                "Basler pose frame IDs must be unique, increasing, 0-based, "
                "and within the saved camera timestamp range."
            )

        matched_exposure_timestamps = exposure_timestamps[frame_indices]

        # head_direction.json must already use the GUI convention: 0 degrees up,
        # positive counter-clockwise. This notebook only applies modulo 360.
        HD_tsd = nap.Tsd(t=matched_exposure_timestamps, d=hd)
        ttl_qc["motive_frame_count_raw"] = int(len(hd_frames))
        ttl_qc["matched_motive_frame_count"] = int(len(hd_frames))
        ttl_qc["frame_alignment_policy_requested"] = "basler_pose_frame_id"
        ttl_qc["frame_alignment_policy_applied"] = "index_by_zero_based_frame_id"
        ttl_qc["frame_timestamp_mapping"] = (
            "zero_based_pose_frame_id_to_saved_camera_timestamp"
        )
        pbar.update(1)

    interval_pairs = interval_pairs_all[[0]]

    hd_tuning_curves_A_formatted = tuning_curve(base_dir=base_dir,
                                                kilosort_dir=kilosort_dir,
                                                probe_name=probe_name,
                                                interval_pairs=interval_pairs,
                                                HD_tsd=HD_tsd,
                                                adc_time_origin_s=adc_time_origin_s,
                                                num_of_bins_in_hd=num_of_bins_in_hd,
                                                num_shuffle=num_shuffle,
                                                shuffle_seed=shuffle_seed,
                                                is_save=True,
                                                metadata={
                                                    "epoch": phase_key,
                                                    "headplate": headplate_name,
                                                    "ttl_qc": ttl_qc,
                                                },
                                                )


Reading files: 100%|██████████| 5/5 [100%]


32


Classifying HD cells: 100%|██████████| 81/81 [00:09<00:00,  8.33unit/s]
